
# **Case Study: Pneumonia Detection Using Fine-Tuned CNN (ResNet50)**
## **Objective**
The goal of this case study is to develop a deep learning model capable of classifying **chest X-ray images** into two categories:  
1. **Normal** (No Pneumonia)  
2. **Pneumonia** (Infected Lungs)  

To achieve this, we use **transfer learning** with the **ResNet50** model, which has been pre-trained on the **ImageNet** dataset. Since medical images differ significantly from natural images, fine-tuning the model helps improve accuracy.

## **Approach**
### **Step 1: Feature Extraction**
- We **freeze** all layers of ResNet50 and use it as a **feature extractor**.
- A **custom classification head** is added on top of ResNet50.
- The model is **trained for 10 epochs** using chest X-ray images.

### **Step 2: Fine-Tuning**
- We **unfreeze the last 50 layers** of ResNet50.
- The model is **fine-tuned for 5 more epochs** with a **lower learning rate** to adapt to medical image features.
- Fine-tuning allows the deeper layers to learn pneumonia-specific patterns.

## **Why Fine-Tuning?**
Fine-tuning is necessary because:
1. Medical images have **different textures** than ImageNet images (which mostly contain natural objects).
2. Some **lower-level features** from ImageNet are useful, but deeper layers must be trained to recognize pneumonia patterns.
3. A **lower learning rate** ensures that the pre-trained knowledge is not lost but adapted.

## **Dataset**
The dataset consists of **chest X-ray images** collected from real-world medical studies. The images are split into:
- **Training Set (80%)**
- **Validation Set (20%)**

The dataset is loaded using **ImageDataGenerator**, which also applies **data augmentation** (rotation, shifting, flipping) to improve generalization.

## **Conclusion**
By leveraging **pre-trained models and fine-tuning**, we enhance model performance for pneumonia detection while reducing training time and dataset size requirements.
"""

## **Imports & Model Loading**

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load pre-trained model (ResNet50) without top layers
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


## **Feature Extraction & Custom Head**

In [ ]:
# Freeze the base model layers (Feature Extraction)
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(2, activation='softmax')  # Modify for pneumonia (2 classes: normal/pneumonia)


## **Model Compilation & Summary**

In [ ]:
# Create new model
model = Model(inputs=base_model.input, outputs=outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print model summary
model.summary()


# **Data Augmentation & Data Loading**

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load training and validation data
train_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


# **Feature Extraction Training**

In [ ]:
# Train the model (Feature Extraction)
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)


# **Fine-Tuning Setup**

In [ ]:
# Unfreeze the last 50 layers for fine-tuning
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Recompile model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


# **Fine-Tuning Training**

In [ ]:
# Fine-tune the model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)


In [ ]:
# Save the fine-tuned model
model.save('fine_tuned_pneumonia_model.h5')
